# TASK-20 · Google Colab Ollama setup

**Run each cell in order** (or Runtime → Run all). Colab is a Linux VM, **not** your Windows computer. The GPU may be unavailable; Ollama can run on CPU. A fresh runtime may require redownloading both models. Scripts in this folder have not been live-tested on Colab.

## Setup: hosted accelerator and dependencies

The next cells inspect Colab's optional GPU, install **Python packages**, install required `zstd` for extraction, then install the official **Ollama Linux server**, start its API, and download two small models. On each notebook's fresh runtime the setup repeats. Official installation script executes remote code; only use the official Ollama domain.

In [ ]:
import shutil
import subprocess

print("NVIDIA GPU utility available:", bool(shutil.which("nvidia-smi")))
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU assigned — expect CPU inference.")

In [ ]:
%pip install -q 'langchain-ollama>=1.0,<2' 'langchain-core>=1.0,<2' 'ollama>=0.6,<1'

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time
import urllib.request

def ollama_ready():
    try:
        with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

if not ollama_ready():
    server_log = open("/tmp/ollama-server.log", "a")
    server = subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(45):
        if ollama_ready():
            break
        time.sleep(2)
assert ollama_ready(), "Ollama failed to start; inspect /tmp/ollama-server.log"
print("Ollama API responding on localhost:11434")

In [ ]:
import subprocess
for model in ("qwen3:1.7b", "embeddinggemma-2:270m"):
    subprocess.run(["ollama", "pull", model], check=True)
subprocess.run(["ollama", "list"], check=True)

## Verify current loading

`ollama ps` only lists models that remain loaded. The list may be empty before inference; this is **not** a failure.

In [ ]:
subprocess.run(["ollama", "ps"], check=True)
print("Ready for TASK-20 lesson notebooks.")